In [1]:
import os
import jsonlines
import numpy as np
import pandas as pd
import pickle
from sklearn.decomposition import PCA

In [2]:
# set your data name
dataset = "trip" # trip, yelp, grocery

In [3]:
def read_data(file_name):
    '''read data from jsonlines file'''
    data = []

    with jsonlines.open(os.path.join(f"{dataset}-semantic-bf16", f"{file_name}.json"), "r") as f:
        for meta_data in f:
            data.append(meta_data)

    return data

In [4]:
# 
file_name = dataset
file_path = dataset + "-semantic-bf16"
data = read_data(file_name)

In [5]:
new_data = {}
for meta_data in data:
    new_data[str(meta_data["item_id"])] = meta_data["hidden_states"]

In [6]:
len(data)

3570

In [7]:
keys = sorted([int(k) for k in new_data.keys()])
print(keys[:20], keys[-20:])
print(f"len(keys) = {len(keys)}")
missing = set(range(min(keys), max(keys)+1)) - set(keys)
print(f"Missing keys: {sorted(list(missing))[:20]}")

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20] [3551, 3552, 3553, 3554, 3555, 3556, 3557, 3558, 3559, 3560, 3561, 3562, 3563, 3564, 3565, 3566, 3567, 3568, 3569, 3570]
len(keys) = 3570
Missing keys: []


In [8]:
llm_emb = []
for i in range(1, len(data)+1):
    llm_emb.append(new_data[str(i)])

In [10]:
llm_emb = np.array(llm_emb)
llm_emb.shape

(3570, 4096)

In [11]:
nan_count = np.isnan(llm_emb).sum()
print(f"在 llm_emb 中总共发现了 {nan_count} 个 NaN 值。")

在 llm_emb 中总共发现了 0 个 NaN 值。


In [12]:
pca = PCA(n_components=1536)
pca_emb = pca.fit_transform(llm_emb)

In [ ]:
import os
pickle.dump(llm_emb, open(os.path.join(file_path, "raw_semantics_embeddings.pkl"), "wb"))
pickle.dump(pca_emb, open(os.path.join(file_path, "semantics_embeddings.pkl"), "wb"))